#### document datastructure


In [24]:
from langchain_core.documents import Document

In [25]:
from typing import Any,List

In [26]:

document_one : Document = Document(
    page_content="Hello, world!", metadata={
        "source": "https://example.com",
        "date_created":"2020-04-98",
        "page":3,
        "author":"Nimesh"

        }
)

In [27]:
print(document_one)

page_content='Hello, world!' metadata={'source': 'https://example.com', 'date_created': '2020-04-98', 'page': 3, 'author': 'Nimesh'}


In [28]:
import os

os.makedirs("../data/textfiles",exist_ok=True)

In [29]:
sample_text = {"../data/textfiles/text_one.txt":"""RAG (Retrieval-Augmented Generation) is a technique that combines information retrieval with text generation. 
Instead of relying only on a model's internal memory, RAG first searches a document collection for relevant context. 
The retrieved passages are then provided to the language model to produce a more accurate and grounded response.

A typical RAG pipeline has five steps:
1. Data ingestion: collect documents from sources like PDFs, web pages, and notes.
2. Chunking: split large documents into smaller text chunks.
3. Embedding: convert each chunk into a vector representation.
4. Indexing: store vectors in a vector database for fast similarity search.
5. Retrieval + generation: find top matching chunks for a user query and generate an answer using those chunks.

RAG is useful for question answering, internal knowledge assistants, customer support bots, and document search systems. 
Its main benefits are reduced hallucinations, better factual accuracy, and easy updates by changing the document store.

Common challenges include poor chunk size, weak embeddings, and missing metadata filters. 
Good practices include cleaning text, storing source metadata, and evaluating retrieval quality with real user questions.
"""}

In [30]:
for filepath,content in sample_text.items():
    with open(filepath,'w',encoding="utf-8") as f:
        f.write(content)

print("file created")

file created


In [31]:
from langchain_community.document_loaders import TextLoader

text_loader = TextLoader(file_path="../data/textfiles/text_one",encoding="utf-8")

document = text_loader.load()
print(document)


[Document(metadata={'source': '../data/textfiles/text_one'}, page_content="RAG (Retrieval-Augmented Generation) is a technique that combines information retrieval with text generation. \nInstead of relying only on a model's internal memory, RAG first searches a document collection for relevant context. \nThe retrieved passages are then provided to the language model to produce a more accurate and grounded response.\n\nA typical RAG pipeline has five steps:\n1. Data ingestion: collect documents from sources like PDFs, web pages, and notes.\n2. Chunking: split large documents into smaller text chunks.\n3. Embedding: convert each chunk into a vector representation.\n4. Indexing: store vectors in a vector database for fast similarity search.\n5. Retrieval + generation: find top matching chunks for a user query and generate an answer using those chunks.\n\nRAG is useful for question answering, internal knowledge assistants, customer support bots, and document search systems. \nIts main bene

In [32]:
from langchain_community.document_loaders import DirectoryLoader,PyPDFLoader

dir_loader = DirectoryLoader(
    path="../data/pdfs",
    glob="*.pdf",
    loader_cls=PyPDFLoader,
    show_progress=True
)

print(dir_loader.load())

100%|██████████| 3/3 [00:00<00:00, 15.15it/s]

[Document(metadata={'producer': 'Skia/PDF m130', 'creator': 'Chromium', 'creationdate': '2026-03-04T08:46:39+00:00', 'title': 'Anschreiben für Bewerbung Nimesh.pdf', 'moddate': '2026-03-04T08:46:39+00:00', 'source': '..\\data\\pdfs\\Anschreiben_für_Bewerbung_Nimesh.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='Essen, 03. M ärz 2026\nInitiativbewerbung als Software-Entwickler\nSehr geehrte Damen und Herren,\nauf der Suche nach einer neuen beru\x00ichen Herausforderung bin ich auf die\nInternetpräsenz Ihres Unternehmens aufmerksam geworden und möchte Ihnen hiermit\nmeine Zusammenarbeit anbieten.\nZuletzt habe ich als Softwareentwickler bei der Werkbank GmbH, einer Digitalagentur aus\nBochum, gearbeitet. Dort programmierte ich in Python und war für die Weiterentwicklung\nvon Websites zuständig. Unter anderem habe ich mit dem Wagtail CM S gearbeitet und viel\nBackend-Entwicklung mit den Frameworks Django, Django REST und HTM X gemacht.\nObwohl ich nur sieben M onate 

In [33]:
from pathlib import Path

In [40]:

def load_documents(data_dir: Path) -> List[Document]:
    documents: List[Document] = []
    len_total_files = len([item for item in data_dir.iterdir()])
    print(f"found {len_total_files} files/objects to process")

    for file_path in data_dir.iterdir():

        if not file_path.is_file():
            print(
                f"{file_path} : is not a file  ! Processing Aborting ..\n",
            )
            continue
        file_ext = ((str(file_path).split("."))[-1]).lower()
        if file_ext not in ["txt", "pdf"]:
            print(f"Wrong file format {file_path} ! Processing Aborting ..\n")
            continue
        print(f"processing: {file_path.name} ")
        try:
            if file_ext == "pdf":
                doctype = PyPDFLoader(file_path=file_path)
                loader = doctype.load()
            elif file_ext == "txt":
                doctype = TextLoader(file_path=file_path, encoding="utf-8")
                loader = doctype.load()
            else:
                continue
        except Exception as e:
            print("Error occured while loading document from !!", file_path, e)
            continue
        for docs in loader:
            docs.metadata.update(
                {
                    "source": str(file_path),
                    "filename": file_path.name,
                    "extension": file_ext,
                }
            )
        documents.extend(loader)
        print(f" ✓ loaded {len(loader)} pages.. \n\n")
    print(f"Total  documents processed ✓ : {len(documents)} ")
    return documents


In [ ]:
data_dir : str = Path("../data")
documents=load_documents(data_dir)

### Text splitting into chunks

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_chunk(
    document: List[Document], chunk_size: int = 2000, chunk_overlap: int = 200
):
    total_chunks = []
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap, length_function=len
    )

    chunks = text_splitter.split_documents(documents=document)
    total_chunks.extend(chunks)
    print(f"Splitted {len(document)} Documents into {len(total_chunks)} chunks ✓ \n")
    return total_chunks

